<a href="https://colab.research.google.com/github/NikhithaMondeddu/Cyber-risk-assesment-and-threat-intelligence-platform/blob/main/cyber_risk_assesment_%26_threat_intelligence_platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!apt-get install -y -qq nmap
!pip install streamlit plotly pandas requests lxml fastapi uvicorn -q

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 119086 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) over (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
from google.colab import userdata
import os


os.environ["VT_API_KEY"] = userdata.get("VT_API_KEY") or ""
os.environ["GMAIL_SENDER"] = userdata.get("GMAIL_SENDER") or ""
os.environ["GMAIL_PASSWORD"] = userdata.get("GMAIL_PASSWORD") or ""
os.environ["GMAIL_RECIPIENT"] = userdata.get("GMAIL_RECIPIENT") or ""


print("VT:", "OK" if os.environ["VT_API_KEY"] else "MISSING")
print("EMAIL:", "OK" if os.environ["GMAIL_SENDER"] else "MISSING")

VT: OK
EMAIL: OK


In [ ]:
import os

for folder in ["modules", "dashboard", "dashboard/pages", "scan_results"]:
    os.makedirs(folder, exist_ok=True)

open("modules/__init__.py","w").close()

In [ ]:
%%writefile modules/scanner.py

import subprocess
import xml.etree.ElementTree as ET
import requests
import os

SCAN_DIR = "scan_results"
os.makedirs(SCAN_DIR, exist_ok=True)

def run_nmap_scan(target):
    xml_file = f"{SCAN_DIR}/{target}.xml"
    subprocess.run(["nmap","-Pn","-sV","-oX",xml_file,target],capture_output=True)
    return xml_file

def parse_nmap_xml(xml_file):
    root = ET.parse(xml_file).getroot()
    rows = []

    for host in root.findall("host"):
        ip = host.find("address").get("addr")

        for port in host.findall(".//port"):
            svc = port.find("service")
            state = port.find("state")

            rows.append({
                "ip": ip,
                "port": port.get("portid"),
                "protocol": port.get("protocol","tcp"),
                "state": state.get("state","unknown") if state is not None else "unknown",
                "service": svc.get("name","unknown") if svc is not None else "unknown",
                "product": svc.get("product","") if svc is not None else "",
                "version": svc.get("version","") if svc is not None else ""
            })
    return rows

def check_virustotal(ip, api_key):
    try:
        r = requests.get(
            f"https://www.virustotal.com/api/v3/ip_addresses/{ip}",
            headers={"x-apikey": api_key}
        )

        if r.status_code != 200:
            return {"malicious_reports":0}

        data = r.json()["data"]["attributes"]["last_analysis_stats"]

        return {
            "malicious_reports": data.get("malicious",0)
        }
    except:
        return {"malicious_reports":0}

Overwriting modules/scanner.py


In [ ]:
%%writefile modules/analyser.py

import pandas as pd

SERVICE_RISK = {
    "telnet": 10,
    "ftp": 8,
    "mysql": 7,
    "ssh": 3,
    "http": 2,
    "https": 1
}

def calculate_risk(row):
    base = SERVICE_RISK.get(row["service"], 2)
    return base + row.get("malicious_reports", 0)

def classify(score):
    if score >= 12:
        return "Critical"
    elif score >= 8:
        return "High"
    elif score >= 5:
        return "Medium"
    else:
        return "Low"

def enrich_dataframe(df, vt_data):

    df = df.copy()

    vt_df = pd.DataFrame(vt_data).T.reset_index().rename(columns={"index":"ip"})
    df = df.merge(vt_df, on="ip", how="left")

    df["malicious_reports"] = df["malicious_reports"].fillna(0)

    df["risk_score"] = df.apply(calculate_risk, axis=1)
    df["severity"] = df["risk_score"].apply(classify)

    return df

Overwriting modules/analyser.py


In [ ]:
%%writefile modules/db.py

import sqlite3
import pandas as pd
from datetime import datetime

DB = "cyberscan.db"

def init_db():
    conn = sqlite3.connect(DB)
    conn.execute("""
    CREATE TABLE IF NOT EXISTS scans(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        scan_time TEXT,
        total_hosts INTEGER,
        total_ports INTEGER,
        max_risk INTEGER,
        results_json TEXT
    )
    """)
    conn.commit()
    conn.close()

def save_scan(df):
    conn = sqlite3.connect(DB)

    conn.execute("""
    INSERT INTO scans(scan_time,total_hosts,total_ports,max_risk,results_json)
    VALUES(?,?,?,?,?)
    """,(
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        df["ip"].nunique(),
        len(df),
        df["risk_score"].max(),
        df.to_json(orient="records")
    ))

    conn.commit()
    conn.close()

def load_history():
    conn = sqlite3.connect(DB)
    df = pd.read_sql_query("SELECT * FROM scans ORDER BY id DESC",conn)
    conn.close()
    return df

def load_scan(id):
    conn = sqlite3.connect(DB)
    row = conn.execute("SELECT results_json FROM scans WHERE id=?",(id,)).fetchone()
    conn.close()

    if row:
        return pd.read_json(row[0])
    return pd.DataFrame()

Overwriting modules/db.py


In [ ]:
import os

print(os.getcwd())


/content


In [ ]:
%%writefile dashboard/app.py


import os
import sys
sys.path.append(os.getcwd())
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import smtplib

from email.mime.multipart import MIMEMultipart
from email.mime.application import MIMEApplication
from email.mime.text import MIMEText

from modules.scanner import *
from modules.analyser import *
from modules.db import *



# CONFIG

st.set_page_config(layout="wide")

init_db()


VT_API_KEY = os.getenv("VT_API_KEY")
GMAIL_SENDER = os.getenv("GMAIL_SENDER")
GMAIL_PASSWORD = os.getenv("GMAIL_PASSWORD")
GMAIL_RECIPIENT = os.getenv("GMAIL_RECIPIENT")

# PDF FUNCTION

from fpdf import FPDF
from datetime import datetime

def generate_pdf(df):

    pdf = FPDF()
    pdf.add_page()

    pdf.set_font("Arial","B",16)
    pdf.cell(0,10,"CyberScan Report",ln=True)

    pdf.set_font("Arial","",10)
    pdf.cell(0,10,f"Generated: {datetime.now()}",ln=True)

    pdf.ln(5)

    for _, row in df.iterrows():
        pdf.cell(0,8,
            f"{row['ip']} | {row['service']} | {row['severity']} | {row['risk_score']}",
            ln=True
        )

    file = "report.pdf"
    pdf.output(file)
    return file


# EMAIL FUNCTION

def send_email_with_pdf(df):

    if not (GMAIL_SENDER and GMAIL_PASSWORD and GMAIL_RECIPIENT):
        return "Email config missing ❌"

    pdf_file = generate_pdf(df)

    msg = MIMEMultipart()
    msg["Subject"] = "🚨 CyberScan Report"
    msg["From"] = GMAIL_SENDER
    msg["To"] = GMAIL_RECIPIENT

    msg.attach(MIMEText("<h3>Cyber Risk Report Attached</h3>", "html"))

    with open(pdf_file,"rb") as f:
        part = MIMEApplication(f.read(), Name="report.pdf")
        part['Content-Disposition'] = 'attachment; filename="report.pdf"'
        msg.attach(part)

    server = smtplib.SMTP("smtp.gmail.com",587)
    server.starttls()
    server.login(GMAIL_SENDER,GMAIL_PASSWORD)
    server.send_message(msg)
    server.quit()

    return "Email Sent with PDF ✅"


# SIDEBAR

st.sidebar.title("🛡️ CyberScan")

menu = st.sidebar.radio(
    "Navigation",
    ["Overview", "Detailed Report", "History", "Alerts"],
    key="nav_main"
)

st.sidebar.markdown("---")


# RUN SCAN

if st.sidebar.button("🚀 Run Scan"):

    TARGETS = ["scanme.nmap.org","testphp.vulnweb.com","testasp.vulnweb.com"]

    with st.spinner("Running scan..."):

        all_rows = []

        for t in TARGETS:
            xml = run_nmap_scan(t)
            all_rows.extend(parse_nmap_xml(xml))

        df = pd.DataFrame(all_rows)

        vt = {ip: check_virustotal(ip, VT_API_KEY) for ip in df["ip"].unique()}
        df = enrich_dataframe(df, vt)

        st.session_state["df"] = df
        save_scan(df)

        # 🔥 AUTO ALERT
        critical = df[df["severity"] == "Critical"]
        if not critical.empty:
            send_email_with_pdf(df)
            st.warning("🚨 Critical issues detected! Email sent.")

    st.success("Scan Completed ✅")


# LOAD DATA

df = st.session_state.get("df")


# 📊 OVERVIEW

if menu == "Overview":

    st.title("📊 Dashboard Overview")

    if df is None:
        st.warning("Run scan first")
    else:
        # KPIs
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Hosts", df["ip"].nunique())
        c2.metric("Findings", len(df))
        c3.metric("Critical", (df["severity"]=="Critical").sum())
        c4.metric("High", (df["severity"]=="High").sum())

        st.markdown("---")

        # 🎯 RISK METER
        avg_risk = df["risk_score"].mean()

        fig_gauge = go.Figure(go.Indicator(
            mode="gauge+number",
            value=avg_risk,
            title={'text': "Risk Meter"},
            gauge={
                'axis': {'range': [0, 15]},
                'bar': {'color': "red"},
                'steps': [
                    {'range': [0, 5], 'color': "green"},
                    {'range': [5, 10], 'color': "yellow"},
                    {'range': [10, 15], 'color': "red"}
                ]
            }
        ))

        st.plotly_chart(fig_gauge, use_container_width=True)

        # ---------------- CHARTS ----------------
        col1, col2, col3 = st.columns(3)

        # Severity Pie
        with col1:
            st.subheader("Severity Distribution")
            fig = px.pie(
                df,
                names="severity",
                color="severity",
                color_discrete_map={
                    "Critical": "#e63946",
                    "High": "#ff9f1c",
                    "Medium": "#ffd166",
                    "Low": "#06d6a0"
                }
            )
            st.plotly_chart(fig, use_container_width=True)

        # Top Hosts
        with col2:
            st.subheader("Top Risky Hosts")
            host_risk = df.groupby("ip")["risk_score"].sum().reset_index()
            fig = px.bar(
                host_risk,
                x="ip",
                y="risk_score",
                color="risk_score",
                color_continuous_scale=["green","yellow","red"]
            )
            st.plotly_chart(fig, use_container_width=True)

        # Service Exposure
        with col3:
            st.subheader("Service Exposure")
            svc = df["service"].value_counts().reset_index()
            svc.columns = ["service", "count"]
            fig = px.bar(svc, x="service", y="count", color="count")
            st.plotly_chart(fig, use_container_width=True)

        st.markdown("---")

        col4, col5, col6 = st.columns(3)

        # Histogram
        with col4:
            st.subheader("Risk Distribution")
            fig = px.histogram(df, x="risk_score", nbins=20)
            st.plotly_chart(fig, use_container_width=True)

        # Threat vs Risk
        with col5:
            st.subheader("Threat vs Risk")
            fig = px.scatter(
                df,
                x="malicious_reports",
                y="risk_score",
                color="severity",
                size="risk_score"
            )
            st.plotly_chart(fig, use_container_width=True)

        # Heatmap
        with col6:
            st.subheader("Attack Surface Heatmap")
            pivot = df.pivot_table(
                index="ip",
                columns="service",
                values="risk_score",
                aggfunc="sum",
                fill_value=0
            )
            fig = px.imshow(pivot, text_auto=True)
            st.plotly_chart(fig, use_container_width=True)

        # 🌐 Threat Intelligence
        st.markdown("---")
        st.subheader("🌐 Threat Intelligence")

        suspicious = df[df["malicious_reports"] > 0]

        if suspicious.empty:
            st.success("No malicious IPs detected ✅")
        else:
            st.error(f"{len(suspicious)} suspicious IPs found")
            st.dataframe(suspicious[["ip","malicious_reports"]])


# 📄 DETAILED REPORT

elif menu == "Detailed Report":

    st.title("📄 Detailed Report")

    if df is None:
        st.warning("Run scan first")
    else:
        st.dataframe(df)

# 📜 HISTORY

elif menu == "History":

    st.title("📜 Scan History")

    hist = load_history()

    if hist.empty:
        st.info("No previous scans")
    else:
        st.dataframe(hist)

        fig = px.line(hist, x="scan_time", y="max_risk", markers=True)
        st.plotly_chart(fig, use_container_width=True)


# 📧 ALERTS

elif menu == "Alerts":

    st.title("📧 Alerts")

    if df is None:
        st.warning("Run scan first")
    else:
        if st.button("📧 Send Email Report"):
            result = send_email_with_pdf(df)
            st.success(result)


Overwriting dashboard/app.py


In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

def send_email():

    if not (GMAIL_SENDER and GMAIL_PASSWORD and GMAIL_RECIPIENT):
        return "Email config missing ❌"

    msg = MIMEMultipart()
    msg["Subject"] = "CyberScan Alert"
    msg["From"] = GMAIL_SENDER
    msg["To"] = GMAIL_RECIPIENT

    body = """
    <h2>🚨 CyberScan Alert</h2>
    <p>Scan completed. Check dashboard for details.</p>
    """

    msg.attach(MIMEText(body, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(GMAIL_SENDER, GMAIL_PASSWORD)
        server.send_message(msg)
        server.quit()

        return "Email sent successfully ✅"

    except Exception as e:
        return f"Error: {e}"


In [ ]:
!pip install fpdf -q

In [ ]:

import os
if not os.path.exists('.streamlit'):
    os.makedirs('.streamlit')

In [ ]:
%%writefile .streamlit/config.toml

[theme]
primaryColor="#ff6b6b"
backgroundColor="#0d1117"
secondaryBackgroundColor="#161b22"
textColor="#c9d1d9"
font="sans serif"

Overwriting .streamlit/config.toml


In [ ]:
!pkill streamlit
!streamlit run dashboard/app.py &>/content/logs.txt &

In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-03-28T01:36:23Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-28T01:36:23Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-28T01:36:27Z INF +--------------------------------------------------------------------------------------------+
2026-03-28T01:36:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-28T01:36:27Z INF |  https://spectrum-tired-refined-wrestling.trycloudflar